In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockA (U) — 读取 U 数据 & 预处理 & 写缓存
#
# 功能：
#   1) 从 EXTR U.zm 文件中，提取 60°N 处的纬向平均风 U(time, lev)
#   2) 从 merged 文件中，同样提取 60°N 处的纬向平均风 U(time, lev)
#   3) 结果写入 NetCDF，供后续绘图 Block 使用
#   4) 极端年信息沿用 O3 目录中的定义
#
# 路径：
#   O3_ROOT = /home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3
#   U_ROOT  = /home/weiji/test_code/plots/New_weiji_general_why0008important/General_U
#
#   EXTR U 输入：
#     /mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/CO2x1SmidEmin_yBWCN.cam.h1.0101-0300_U.zm.nc
#   merged 输入：
#     /mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc
#
#   输出：
#     U_EXTR_60N_lev_time.nc   — U_60N(time, lev) [m/s]
#     U_MERGED_60N_lev_time.nc — U_60N(time, lev) [m/s]
# ============================================================

import os
import numpy as np
import xarray as xr

# 目录设定
O3_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
U_ROOT  = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_U"
os.makedirs(U_ROOT, exist_ok=True)

# 输入文件路径 (假设 U 的 zm 文件命名规则与 T 一致)
EXTR_U_PATH = (
    "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/"
    "CO2x1SmidEmin_yBWCN.cam.h1.0101-0300_U.zm.nc"
)
MERGED_PATH = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.merged.nc"
)

LOW10_TXT = os.path.join(O3_ROOT, "O3_lowest10_years.txt")
LOW25_TXT = os.path.join(O3_ROOT, "O3_lowest25pct_years.txt")

# 输出文件路径 (对应你 ls 结果中的 .nc 文件)
U_EXTR_60N_NC   = os.path.join(U_ROOT, "U_EXTR_60N_lev_time.nc")
U_MERGED_60N_NC = os.path.join(U_ROOT, "U_MERGED_60N_lev_time.nc")


def calc_U60N(ds, var="U"):
    """
    提取 60°N 附近的纬向平均风 U(time, lev):

      - 若有 lon -> 先经向平均 (Zonal Mean)
      - 选取最靠近 60°N 的纬度值
      - 垂直维统一命名为 'lev'，单位统一为 Pa

    返回 DataArray: U_60N(time, lev) [m/s]
    """
    da = ds[var]

    # zonal mean (处理非 zm 文件的情况)
    if "lon" in da.dims:
        da = da.mean(dim="lon")

    if "lat" not in da.dims:
        raise ValueError(f"{var} has no 'lat' dimension, dims={da.dims}")

    # 选取 60N (这里使用 method="nearest" 以防网格不恰好在 60.0)
    da_60N = da.sel(lat=60.0, method="nearest")

    # 识别垂直维
    lev_name = None
    for name in ("plev", "lev_p", "lev", "level"):
        if name in da_60N.dims:
            lev_name = name
            break
    if lev_name is None:
        raise ValueError(
            f"{var}_60N has no vertical dim, dims={da_60N.dims}"
        )

    lev_vals = da_60N[lev_name].values
    max_lev = float(np.nanmax(lev_vals))

    # 统一压力单位为 Pa
    if max_lev <= 2000.0:
        lev_pa = lev_vals * 100.0
    else:
        lev_pa = lev_vals

    da_60N = da_60N.rename({lev_name: "lev"})
    da_60N = da_60N.assign_coords(lev=("lev", lev_pa))

    return da_60N  # (time, lev)


def main_blockA_U():
    # ==== 检查 O3 极端年列表 ====
    if not (os.path.exists(LOW10_TXT) and os.path.exists(LOW25_TXT)):
        print(f"[Warning] O3 files not found in {O3_ROOT}. Please check if paths are correct.")
    else:
        lowest10_years = np.loadtxt(LOW10_TXT, dtype=int).reshape(-1)
        print("[BlockA_U] Lowest-10 O3 years detected:", lowest10_years)

    # ==== 1. EXTR U_60N(time, lev) ====
    print("[BlockA_U] Reading EXTR U.zm:", EXTR_U_PATH)
    try:
        ds_extr = xr.open_dataset(EXTR_U_PATH)
        u_extr = calc_U60N(ds_extr, var="U")
        ds_extr.close()

        u_extr = u_extr.transpose("time", "lev")
        u_extr.name = "U_60N"
        u_extr.attrs["long_name"] = "Zonal mean zonal wind at 60N"
        u_extr.attrs["units"] = "m/s"
        u_extr["lev"].attrs["units"] = "Pa"

        u_extr.to_netcdf(U_EXTR_60N_NC)
        print("[BlockA_U] Saved EXTR U_60N(time,lev) to:", U_EXTR_60N_NC)
    except FileNotFoundError:
        print(f"[Error] Could not find {EXTR_U_PATH}")

    # ==== 2. merged U_60N(time, lev) ====
    print("[BlockA_U] Reading merged file:", MERGED_PATH)
    try:
        ds_merged = xr.open_dataset(MERGED_PATH)
        u_merged = calc_U60N(ds_merged, var="U")
        ds_merged.close()

        u_merged = u_merged.transpose("time", "lev")
        u_merged.name = "U_60N"
        u_merged.attrs["long_name"] = "Zonal mean zonal wind at 60N"
        u_merged.attrs["units"] = "m/s"
        u_merged["lev"].attrs["units"] = "Pa"

        u_merged.to_netcdf(U_MERGED_60N_NC)
        print("[BlockA_U] Saved merged U_60N(time,lev) to:", U_MERGED_60N_NC)
    except FileNotFoundError:
        print(f"[Error] Could not find {MERGED_PATH}")

    print("[BlockA_U] Done.")


if __name__ == "__main__":
    main_blockA_U()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockB (U) — 60°N 纬向风折线图（1 / 10 / 50 / 100 hPa）
#
# 功能：
#   1) EXTR：10 个低 O3 年 vs 其它年份 (U_60N at diff levels)
#   2) merged：0008/0013/0014/0019 vs EXTR 气候态 (all / low25)
#
# 时间轴：前一年 Oct–Dec (91 天) + 当年 Jan–Sep，x=0..364
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import rcParams

O3_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
U_ROOT  = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_U"
os.makedirs(U_ROOT, exist_ok=True)

U_EXTR_60N_NC   = os.path.join(U_ROOT, "U_EXTR_60N_lev_time.nc")
U_MERGED_60N_NC = os.path.join(U_ROOT, "U_MERGED_60N_lev_time.nc")
LOW10_TXT       = os.path.join(O3_ROOT, "O3_lowest10_years.txt")
LOW25_TXT       = os.path.join(O3_ROOT, "O3_lowest25pct_years.txt")

N_PREV = 91
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)

def get_level_index(lev_vals_pa, target_hpa):
    target_pa = target_hpa * 100.0
    idx = int(np.argmin(np.abs(lev_vals_pa - target_pa)))
    return idx, float(lev_vals_pa[idx] / 100.0)

def plot_extremes_vs_others_U(var_extr, lowest10_years, level_label, out_png, out_pdf):
    years_extr = np.unique(var_extr.time.dt.year.values.astype(int))
    n_days = int(var_extr.time.dt.dayofyear.max())

    fig, ax = plt.subplots(1, 1, figsize=(13, 10), constrained_layout=True)

    low_years = lowest10_years
    neutral_years = years_extr[~np.isin(years_extr, low_years)]

    # 提取数据
    var_low_cur = var_extr.sel(time=var_extr.time.dt.year.isin(low_years))
    var_low_prev = var_extr.sel(time=var_extr.time.dt.year.isin(low_years - 1))
    var_neu_cur = var_extr.sel(time=var_extr.time.dt.year.isin(neutral_years))
    var_neu_prev = var_extr.sel(time=var_extr.time.dt.year.isin(neutral_years - 1))

    # 计算气候态
    neu_cur_mean = np.array(var_neu_cur.groupby("time.dayofyear").mean("time"))
    neu_cur_std  = np.array(var_neu_cur.groupby("time.dayofyear").std("time"))
    neu_prev_mean = np.array(var_neu_prev.groupby("time.dayofyear").mean("time"))
    neu_prev_std  = np.array(var_neu_prev.groupby("time.dayofyear").std("time"))

    n_extreme = len(low_years)
    var_low_cur_arr  = np.reshape(np.array(var_low_cur), (n_extreme, n_days))
    var_low_prev_arr = np.reshape(np.array(var_low_prev), (n_extreme, n_days))

    colors_ext = cm.twilight(np.linspace(0, 1, n_extreme))

    for j in range(n_extreme):
        ax.plot(range(N_PREV, n_days), var_low_cur_arr[j, 0:n_days - N_PREV],
                color=colors_ext[j], alpha=0.8, linewidth=2, label="low O3 years" if j == 0 else None)
        ax.plot(range(0, N_PREV), var_low_prev_arr[j, n_days - N_PREV:n_days],
                color=colors_ext[j], alpha=0.8, linewidth=2)

    ax.errorbar(range(N_PREV, n_days), neu_cur_mean[0:n_days - N_PREV],
                neu_cur_std[0:n_days - N_PREV], color="grey", alpha=0.5, linewidth=3, label="all other years")
    ax.errorbar(range(0, N_PREV), neu_prev_mean[n_days - N_PREV:n_days],
                neu_prev_std[n_days - N_PREV:n_days], color="grey", alpha=0.5, linewidth=3)

    # 格式化
    ax.set_xlim(0, n_days)
    ax.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
    ax.set_xticklabels(["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "June", "July", "Aug", "Sep"], fontsize=15)
    ax.set_ylabel(f"U at 60°N, {level_label} (m/s)", fontsize=18)
    ax.legend(fontsize=14)
    ax.set_title(f"Zonal Mean Zonal Wind (60°N), {level_label} — low O$_3$ years vs others")
    
    # 参考线：0 m/s (SSW threshold)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1.0, alpha=0.5)

    plt.savefig(out_png, dpi=300); plt.savefig(out_pdf); plt.close(fig)

def plot_special_vs_climatology_U(var_extr, var_merged, lowest25_years, level_label, out_png, out_pdf):
    n_days = int(var_extr.time.dt.dayofyear.max())
    
    # EXTR Climatology
    clim_all_mean = np.array(var_extr.groupby("time.dayofyear").mean("time"))
    clim_all_std  = np.array(var_extr.groupby("time.dayofyear").std("time"))

    # EXTR Low-25% Composite
    base_low25_cur = var_extr.sel(time=var_extr.time.dt.year.isin(lowest25_years))
    base_low25_prev = var_extr.sel(time=var_extr.time.dt.year.isin(lowest25_years - 1))
    
    low25_c_m = np.array(base_low25_cur.groupby("time.dayofyear").mean("time"))
    low25_c_s = np.array(base_low25_cur.groupby("time.dayofyear").std("time"))
    low25_p_m = np.array(base_low25_prev.groupby("time.dayofyear").mean("time"))
    low25_p_s = np.array(base_low25_prev.groupby("time.dayofyear").std("time"))

    # 拼接跨年序列 (Oct-Sep)
    def concat_oct_sep(prev_data, cur_data):
        return np.concatenate([prev_data[n_days - N_PREV:n_days], cur_data[0:n_days - N_PREV]])

    all_comp_mean = concat_oct_sep(clim_all_mean, clim_all_mean)
    all_comp_std  = concat_oct_sep(clim_all_std, clim_all_std)
    low25_comp_mean = concat_oct_sep(low25_p_m, low25_c_m)
    low25_comp_std  = concat_oct_sep(low25_p_s, low25_c_s)

    # 绘图配置
    rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
    fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.0), constrained_layout=True)
    x_full = np.arange(n_days)
    colors_spec = plt.cm.tab10(np.linspace(0, 1, len(YEARS_SPECIAL)))
    
    handles_years = []
    years_merged = np.unique(var_merged.time.dt.year.values.astype(int))

    for i, y in enumerate(YEARS_SPECIAL):
        if y in years_merged and (y-1) in years_merged:
            cur_arr = np.array(var_merged.sel(time=var_merged.time.dt.year == y))
            prev_arr = np.array(var_merged.sel(time=var_merged.time.dt.year == (y-1)))
            series = concat_oct_sep(prev_arr, cur_arr)
            ax.plot(x_full, series, label=f"Year {int(y):04d}", color=colors_spec[i], lw=1.5)
            handles_years.append(Line2D([0], [0], color=colors_spec[i], lw=1.5, label=f"Year {int(y):04d}"))

    # 阴影与气候态
    ax.fill_between(x_full, all_comp_mean - all_comp_std, all_comp_mean + all_comp_std, color="0.85", alpha=0.9, zorder=0)
    ax.plot(x_full, all_comp_mean, "k--", lw=1.8, label="EXTR all years")
    ax.fill_between(x_full, low25_comp_mean - low25_comp_std, low25_comp_mean + low25_comp_std, facecolor="none", edgecolor="0.5", hatch="///", zorder=1)
    ax.plot(x_full, low25_comp_mean, "k-", lw=2.0, label="EXTR low-O3 composite")

    ax.set_xlim(0, n_days); ax.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
    ax.set_xticklabels(["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "June", "July", "Aug", "Sep"])
    ax.set_ylabel(f"U at 60°N, {level_label} (m/s)")
    ax.axhline(y=0, color='black', lw=0.8, alpha=0.5)

    handles = handles_years + [
        Line2D([0], [0], color="black", lw=1.8, ls="--", label="EXTR all-year"),
        Patch(facecolor="0.85", label="EXTR all-year ±1σ"),
        Line2D([0], [0], color="black", lw=2.0, ls="-", label="EXTR low-O3"),
        Patch(facecolor="none", edgecolor="0.5", hatch="///", label="EXTR low-O3 ±1σ")
    ]
    ax.legend(handles=handles, loc="best", frameon=False, fontsize=8, ncol=2)
    ax.set_title(f"Zonal Mean Zonal Wind (60°N), {level_label}")

    plt.savefig(out_png, dpi=300); plt.savefig(out_pdf); plt.close(fig)

def main_blockB_U():
    u_extr = xr.load_dataarray(U_EXTR_60N_NC)
    u_merged = xr.load_dataarray(U_MERGED_60N_NC)
    lev_vals = u_extr["lev"].values
    low10 = np.loadtxt(LOW10_TXT, dtype=int).reshape(-1)
    low25 = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)

    # 对应你 ls 里的四个层次: 1, 10, 50, 100 hPa
    target_levels = [1.0, 10.0, 50.0, 100.0]

    for target_hpa in target_levels:
        idx, lev_act = get_level_index(lev_vals, target_hpa)
        lev_int = int(round(lev_act))
        label = f"{lev_act:.1f} hPa"
        
        v_ext = u_extr.isel(lev=idx)
        v_mer = u_merged.isel(lev=idx)

        # 匹配你的文件名格式
        # 1. Evolution 图
        png1 = os.path.join(U_ROOT, f"U_60N_{lev_int}hPa_evolution_min_10extr_200yrs.png")
        pdf1 = os.path.join(U_ROOT, f"U_60N_{lev_int}hPa_evolution_min_10extr_200yrs.pdf")
        plot_extremes_vs_others_U(v_ext, low10, label, png1, pdf1)

        # 2. Daily comparison 图
        png2 = os.path.join(U_ROOT, f"U_60N_{lev_int}hPa_daily_special_years_vs_extr_climatology.png")
        pdf2 = os.path.join(U_ROOT, f"U_60N_{lev_int}hPa_daily_special_years_vs_extr_climatology.pdf")
        plot_special_vs_climatology_U(v_ext, v_mer, low25, label, png2, pdf2)

    print("[BlockB_U] All plots generated.")

if __name__ == "__main__":
    main_blockB_U()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockC (U) — 60°N U 垂直时间剖面 anomaly + 显著性
#
# 输出 NetCDF:
#   U_vert_anom_special_60N.nc
#
# variables (m/s):
#   anom_all, anom_low25, clim_all_comp, clim_low25_comp
#
# significance mask:
#   t_sig_all, b_sig_all, t_sig_low25, b_sig_low25
# ============================================================

import os
import numpy as np
import xarray as xr
from scipy.stats import t as t_dist

O3_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
U_ROOT  = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_U"
os.makedirs(U_ROOT, exist_ok=True)

EXTR_U_PATH = (
    "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/"
    "CO2x1SmidEmin_yBWCN.cam.h1.0101-0300_U.zm.nc"
)
MERGED_FILE = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.merged.nc"
)

LOW25_TXT = os.path.join(O3_ROOT, "O3_lowest25pct_years.txt")
OUT_NC = os.path.join(U_ROOT, "U_vert_anom_special_60N.nc")

N_PREV = 91
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)

def calc_U60N_pcap(ds, var="U"):
    """ 提取 60°N 处的纬向平均风 U(time, lev) """
    da = ds[var]
    if "lon" in da.dims:
        da = da.mean(dim="lon")
    
    da_60N = da.sel(lat=60.0, method="nearest")

    lev_name = None
    for name in ("plev", "lev_p", "lev", "level"):
        if name in da_60N.dims:
            lev_name = name
            break
            
    lev_vals = da_60N[lev_name].values
    lev_pa = lev_vals * 100.0 if np.nanmax(lev_vals) <= 2000.0 else lev_vals
    
    da_60N = da_60N.rename({lev_name: "lev"})
    da_60N = da_60N.assign_coords(lev=("lev", lev_pa))
    return da_60N

def bootstrap_ci(data, nboot=5000, alpha=0.05, rng=None):
    data = np.asarray(data)
    data = data[~np.isnan(data)]
    n = data.size
    if n < 2: return np.nan, np.nan
    if rng is None: rng = np.random.default_rng()
    
    means = np.empty(nboot)
    for i in range(nboot):
        samp = rng.choice(data, size=n, replace=True)
        means[i] = np.mean(samp)
    
    return np.percentile(means, [100 * alpha/2.0, 100 * (1 - alpha/2.0)])

def compute_significance_for_baseline(base_anom, anom_ref, doy_base, doy_comp, alpha=0.05, nboot=5000):
    base_vals = base_anom.values  # (time, lev)
    lev_n, t_len = base_vals.shape[1], anom_ref.shape[1]
    t_sig = np.zeros((lev_n, t_len), dtype=bool)
    b_sig = np.zeros((lev_n, t_len), dtype=bool)
    rng = np.random.default_rng(2025)

    for ti in range(t_len):
        day = int(doy_comp[ti])
        mask_day = (doy_base == day)
        if not np.any(mask_day): continue
        day_samples = base_vals[mask_day, :]

        for li in range(lev_n):
            col = day_samples[:, li]
            col = col[~np.isnan(col)]
            if col.size < 2: continue

            obs = anom_ref[li, ti]
            se = np.std(col, ddof=1) / np.sqrt(col.size)
            if se == 0: continue
            
            # T-test
            tstat = obs / se
            pval = 2 * (1 - t_dist.cdf(abs(tstat), df=col.size - 1))
            t_sig[li, ti] = (pval < alpha)

            # Bootstrap
            lo, hi = bootstrap_ci(col, nboot=nboot, alpha=alpha, rng=rng)
            if not np.isnan(lo):
                b_sig[li, ti] = not (lo <= obs <= hi)
    return t_sig, b_sig

def main_blockC_U():
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)

    print("[BlockC_U] Loading EXTR U...")
    ds_extr = xr.open_dataset(EXTR_U_PATH)
    u_extr = calc_U60N_pcap(ds_extr, "U")
    ds_extr.close()

    doy_extr = u_extr.time.dt.dayofyear.values.astype(int)
    n_days = int(u_extr.time.dt.dayofyear.max())

    # 1. Climatology construction
    clim_all = u_extr.groupby("time.dayofyear").mean("time")
    
    base_low25_cur = u_extr.sel(time=u_extr.time.dt.year.isin(lowest25_years))
    base_low25_prev = u_extr.sel(time=u_extr.time.dt.year.isin(lowest25_years - 1))
    
    clim_low25_cur = base_low25_cur.groupby("time.dayofyear").mean("time")
    clim_low25_prev = base_low25_prev.groupby("time.dayofyear").mean("time")
    
    clim_low25_full = clim_low25_cur.copy()
    oct_dec = np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int)
    clim_low25_full.loc[dict(dayofyear=oct_dec)] = clim_low25_prev.sel(dayofyear=oct_dec).values

    print("[BlockC_U] Loading merged U...")
    ds_merged = xr.open_dataset(MERGED_FILE)
    u_merged = calc_U60N_pcap(ds_merged, "U")
    ds_merged.close()

    lev = u_merged["lev"].values
    lev_n = lev.size
    doy_comp = np.concatenate([np.arange(n_days - N_PREV + 1, n_days + 1), np.arange(1, n_days - N_PREV + 1)])
    t_len = doy_comp.size

    # 2. Baseline anomalies
    base_anom_all = u_extr - clim_all.sel(dayofyear=u_extr.time.dt.dayofyear)
    base_anom_low25 = u_extr - clim_low25_full.sel(dayofyear=u_extr.time.dt.dayofyear)

    clim_all_comp_vals = clim_all.sel(dayofyear=doy_comp).transpose("lev", "dayofyear").values
    clim_low25_comp_vals = clim_low25_full.sel(dayofyear=doy_comp).transpose("lev", "dayofyear").values

    # 3. Process special years
    n_ref = len(YEARS_SPECIAL)
    res_shape = (n_ref, lev_n, t_len)
    anom_all_arr, anom_low_arr = np.zeros(res_shape), np.zeros(res_shape)
    ts_all, bs_all = np.zeros(res_shape, dtype=bool), np.zeros(res_shape, dtype=bool)
    ts_low, bs_low = np.zeros(res_shape, dtype=bool), np.zeros(res_shape, dtype=bool)

    for yi, y in enumerate(YEARS_SPECIAL):
        if y not in u_merged.time.dt.year: continue
        
        ref_cur = u_merged.sel(time=u_merged.time.dt.year == y).transpose("time", "lev").values
        ref_prev = u_merged.sel(time=u_merged.time.dt.year == (y - 1)).transpose("time", "lev").values
        
        ref_comp = np.concatenate([ref_prev[n_days - N_PREV:n_days, :], ref_cur[0:n_days - N_PREV, :]], axis=0).T
        
        anom_all = ref_comp - clim_all_comp_vals
        anom_low = ref_comp - clim_low25_comp_vals
        
        anom_all_arr[yi] = anom_all
        anom_low_arr[yi] = anom_low

        print(f"[BlockC_U] Significance for Year {y:04d}...")
        ts_all[yi], bs_all[yi] = compute_significance_for_baseline(base_anom_all, anom_all, doy_extr, doy_comp)
        ts_low[yi], bs_low[yi] = compute_significance_for_baseline(base_anom_low25, anom_low, doy_extr, doy_comp)

    # 4. Save
    ds_out = xr.Dataset(
        coords={"ref_year": YEARS_SPECIAL, "lev": lev, "time_index": np.arange(t_len), "dayofyear": ("time_index", doy_comp)},
        data_vars={
            "anom_all": (("ref_year", "lev", "time_index"), anom_all_arr),
            "anom_low25": (("ref_year", "lev", "time_index"), anom_low_arr),
            "clim_all_comp": (("lev", "time_index"), clim_all_comp_vals),
            "clim_low25_comp": (("lev", "time_index"), clim_low25_comp_vals),
            "t_sig_all": (("ref_year", "lev", "time_index"), ts_all),
            "b_sig_all": (("ref_year", "lev", "time_index"), bs_all),
            "t_sig_low25": (("ref_year", "lev", "time_index"), ts_low),
            "b_sig_low25": (("ref_year", "lev", "time_index"), bs_low),
        }
    )
    ds_out.attrs["description"] = "U at 60N anomalies and significance for special years."
    ds_out.to_netcdf(OUT_NC)
    print(f"[BlockC_U] Saved to: {OUT_NC}")

if __name__ == "__main__":
    main_blockC_U()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockD (U) — 绘制 60°N 纬向风时间×高度 anomaly 图
#
# 使用 BlockC_U 输出：
#   U_vert_anom_special_60N.nc
#
# 横坐标：Oct–Sep (0..364)
# 纵坐标：Pressure (log scale, 1000–1 hPa)
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

U_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_U"
os.makedirs(U_ROOT, exist_ok=True)

IN_NC = os.path.join(U_ROOT, "U_vert_anom_special_60N.nc")

XTICKS = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "June", "July", "Aug", "Sep"]

# 绘图基础样式设定
mpl.rc("xtick", direction="out", labelsize=8)
mpl.rc("ytick", direction="out", labelsize=8)
mpl.rc("font", family="sans-serif")
mpl.rc("axes", titlesize=11, labelsize=9)
mpl.rc("figure", figsize=(6, 4), dpi=300)

def guess_p_2_pa(lev_vals):
    lev_vals = np.asarray(lev_vals)
    maxv = np.nanmax(lev_vals)
    if maxv <= 2000.0:
        return lev_vals * 100.0, "hPa"
    else:
        return lev_vals, "Pa"

def plot_time_height_anom_U(
    x_idx, lev_vals, anom_vals, clim_vals, nonsig_mask, 
    ref_year, baseline_label, test_label, outfile
):
    fig, ax = plt.subplots()
    x = np.asarray(x_idx)
    p_pa, p_unit = guess_p_2_pa(lev_vals)

    # U 异常值的范围通常比温度大，设定 vmax=40 m/s
    vlim = np.nanmax(np.abs(anom_vals))
    vmax = min(vlim, 40.0) if np.isfinite(vlim) and vlim > 0 else 20.0
    levels = np.linspace(-vmax, vmax, 31)

    # 填色图 (Anomaly)
    cf = ax.contourf(
        x, p_pa, anom_vals,
        levels=levels, cmap="RdBu_r", extend="both", alpha=0.85
    )

    # 等值线 (Climatology)
    # 绘制 0 线（黑粗线）表示风向切换
    ax.contour(x, p_pa, clim_vals, levels=[0], colors="k", linewidths=1.2)
    # 其它气候态等值线
    CS = ax.contour(x, p_pa, clim_vals, levels=10, colors="k", linewidths=0.5, alpha=0.5)
    ax.clabel(CS, inline=True, fontsize=6, fmt="%d")

    # 显著性阴影（非显著区域打排线）
    if nonsig_mask is not None:
        ax.contourf(
            x, p_pa, nonsig_mask.astype(int),
            levels=[0.5, 1.5], colors="none", hatches=["///"], alpha=0
        )
        patch = Patch(facecolor="white", hatch="///", edgecolor="black", label=f"p ≥ 0.05 ({test_label})")
        ax.legend(handles=[patch], loc="upper right")

    # 坐标轴设置
    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.set_ylabel(f"Pressure ({p_unit})")
    ax.set_xlim(x[0], x[-1])
    ax.set_xticks(XTICKS)
    ax.set_xticklabels(XTICK_LABELS)
    ax.grid(True, which="major", linestyle="--", linewidth=0.4, alpha=0.3)

    ax.set_title(f"U anomaly (m/s) at 60N, Year {int(ref_year):04d}\nRef: {baseline_label}")
    
    cbar = plt.colorbar(cf, ax=ax, pad=0.02)
    cbar.set_label("U anomaly (m/s)")

    plt.savefig(outfile, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {os.path.basename(outfile)}")

def main_blockD_U():
    if not os.path.exists(IN_NC):
        print(f"Error: {IN_NC} not found. Run BlockC first.")
        return

    ds = xr.open_dataset(IN_NC)
    ref_years = ds["ref_year"].values
    lev = ds["lev"].values
    time_idx = ds["time_index"].values

    for yi, y in enumerate(ref_years):
        year_str = f"year{int(y):04d}"
        
        # 方案 1: vs ALL baseline
        # t-test
        plot_time_height_anom_U(
            time_idx, lev, ds["anom_all"][yi].values, ds["clim_all_comp"].values, 
            ~ds["t_sig_all"][yi].values, y, "EXTR All-years", "t-test",
            os.path.join(U_ROOT, f"U_anom_60N_{year_str}_vsALL_t_nonsig.png")
        )
        # bootstrap
        plot_time_height_anom_U(
            time_idx, lev, ds["anom_all"][yi].values, ds["clim_all_comp"].values, 
            ~ds["b_sig_all"][yi].values, y, "EXTR All-years", "bootstrap",
            os.path.join(U_ROOT, f"U_anom_60N_{year_str}_vsALL_bootstrap_nonsig.png")
        )

        # 方案 2: vs LOW25 baseline
        # t-test
        plot_time_height_anom_U(
            time_idx, lev, ds["anom_low25"][yi].values, ds["clim_low25_comp"].values, 
            ~ds["t_sig_low25"][yi].values, y, "EXTR Low-O3", "t-test",
            os.path.join(U_ROOT, f"U_anom_60N_{year_str}_vsLOW25_t_nonsig.png")
        )
        # bootstrap
        plot_time_height_anom_U(
            time_idx, lev, ds["anom_low25"][yi].values, ds["clim_low25_comp"].values, 
            ~ds["b_sig_low25"][yi].values, y, "EXTR Low-O3", "bootstrap",
            os.path.join(U_ROOT, f"U_anom_60N_{year_str}_vsLOW25_bootstrap_nonsig.png")
        )

    ds.close()
    print("[BlockD_U] Done.")

if __name__ == "__main__":
    main_blockD_U()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
#这里的东西在EHE23里面
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

U_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_U"
IN_NC = os.path.join(U_ROOT, "U_vert_anom_special_60N.nc")

# 绘图常量设置
XTICKS = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "June", "July", "Aug", "Sep"]

def plot_u_regimes_plan1(ds, ref_year_val, target_hpa=100.0):
    """复原截图中的 Zonal Wind Regimes 绘图逻辑"""
    
    # 1. 提取数据
    # 找到最接近的目标层索引
    lev_pa = ds.lev.values
    idx = int(np.argmin(np.abs(lev_pa - target_hpa * 100.0)))
    
    # 选取对应年份的 index
    y_idx = int(np.where(ds.ref_year == ref_year_val)[0][0])
    
    # 获取原始 U (通过异常值 + 气候态还原)
    # 截图显示的是 15d RM，我们先取日数据
    clim_mean = ds.clim_all_comp.isel(lev=idx)
    anom = ds.anom_all.isel(ref_year=y_idx, lev=idx)
    u_raw = clim_mean + anom
    
    # 计算 15d Running Mean
    # 注意：为了平滑，使用 rolling，center=True
    u_rm = u_raw.rolling(time_index=15, center=True).mean()
    
    # 2. 识别阶段 (Regimes)
    # 根据 U_RM 的导数判定：连续增加为加强，连续减少为减弱
    u_grad = np.gradient(u_rm.fillna(u_raw).values)
    # 设定微小阈值防止噪声
    threshold = 0.05 
    strengthening = u_grad > threshold
    weakening = u_grad < -threshold
    
    # 3. 开始绘图
    fig, ax1 = plt.subplots(figsize=(11, 6), dpi=300)
    ax2 = ax1.twinx() # 右轴用于 Anomaly
    
    x = ds.time_index.values
    
    # 绘制背景阴影 (Regime background)
    # 遍历时间轴，填充颜色
    for i in range(len(x)-1):
        if strengthening[i]:
            ax1.axvspan(x[i], x[i+1], color='skyblue', alpha=0.25, lw=0)
        elif weakening[i]:
            ax1.axvspan(x[i], x[i+1], color='pink', alpha=0.3, lw=0)

    # 绘制气候态背景阴影 (±1 sigma 范围)
    # 注意：BlockC 中未存 std，这里用示意性灰色带，或你可以根据 BlockB 逻辑补充
    ax1.fill_between(x, clim_mean-5, clim_mean+5, color='lightgrey', alpha=0.4, label='Climatology')

    # 左轴：绘制 Wind (15d RM)
    l1, = ax1.plot(x, u_rm, color='tab:blue', lw=2.5, label='Wind (15d RM)')
    
    # 右轴：绘制 Anomaly (紫色虚线)
    l2, = ax2.plot(x, anom, color='purple', ls=':', lw=1.5, label='Anomaly')
    
    # 4. 细节调整 (匹配截图)
    ax1.set_title(f"Zonal Wind Regimes (Year {int(ref_year_val):04d})\nOriginal Plan 1 Logic + Anomaly", 
                  fontsize=14, fontweight='bold')
    
    ax1.set_ylabel(f"Zonal Wind U (m/s) @ {int(target_hpa)}hPa", color='tab:blue', fontsize=12)
    ax2.set_ylabel("Wind Anomaly (m/s)", color='purple', fontsize=12)
    
    ax1.set_xlim(x[0], x[-1])
    ax1.set_xticks(XTICKS)
    ax1.set_xticklabels(XTICK_LABELS)
    ax1.axhline(y=0, color='grey', lw=0.8, ls='-') # 0线
    
    # 自定义图例 (匹配截图右上角)
    legend_elements = [
        Line2D([0], [0], color='tab:blue', lw=2, label='Wind (15d RM)'),
        Line2D([0], [0], color='purple', ls=':', lw=1.5, label='Anomaly'),
        Line2D([0], [0], color='grey', ls='--', label='Climatology'),
        Patch(facecolor='pink', alpha=0.5, label='Weakening'),
        Patch(facecolor='skyblue', alpha=0.5, label='Strengthening')
    ]
    ax1.legend(handles=legend_elements, loc='upper right', ncol=3, fontsize=9)

    # 5. 保存文件
    file_name = f"U_60N_{int(target_hpa)}hPa_Plan1_Original_Colored.png"
    save_path = os.path.join(U_ROOT, file_name)
    plt.savefig(save_path, bbox_inches='tight')
    plt.savefig(save_path.replace('.png', '.pdf'), bbox_inches='tight')
    plt.close()
    print(f"[BlockE_U] Saved: {file_name}")

def main_blockE():
    if not os.path.exists(IN_NC):
        print(f"Error: {IN_NC} not found.")
        return
        
    ds = xr.open_dataset(IN_NC)
    
    # 针对 Year 0008 绘制 100hPa 的图
    # 你也可以循环 ref_years 和不同层次
    try:
        plot_u_regimes_plan1(ds, ref_year_val=8, target_hpa=100.0)
    except Exception as e:
        print(f"Error plotting Year 0008: {e}")
        
    ds.close()
    print("[BlockE_U] Done.")

if __name__ == "__main__":
    main_blockE()